In [1]:
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
# from sklearn.impute import SimpleImputer
# from sklearn.preprocessing import StandardScaler, OneHotEncoder
# from sklearn.feature_selection import SelectKBest, mutual_info_classif, f_regression
# from sklearn.model_selection import train_test_split, GridSearchCV
# from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
# from sklearn.neural_network import MLPClassifier, MLPRegressor
# from sklearn.metrics import log_loss, mean_absolute_error

In [ ]:
train_path = "./data/AppML_InitialProject_train.h5"

df = pd.read_hdf(train_path)

# for i in df.columns:
#     print(i)

columns_list = sorted([i for i in df.columns])

mask = 'pX_'
[i for i in columns_list if mask in i]
[i for i in columns_list if mask not in i]

['averageInteractionsPerCrossing',
 'p_Eratio',
 'p_Reta',
 'p_Rhad',
 'p_Rhad1',
 'p_Rphi',
 'p_TRTPID',
 'p_TRTTrackOccupancy',
 'p_Truth_Energy',
 'p_Truth_isElectron',
 'p_charge',
 'p_d0',
 'p_dPOverP',
 'p_deltaEta1',
 'p_deltaPhiRescaled2',
 'p_eta',
 'p_etcone20',
 'p_etcone30',
 'p_etcone40',
 'p_f1',
 'p_f3',
 'p_numberOfInnermostPixelHits',
 'p_numberOfPixelHits',
 'p_numberOfSCTHits',
 'p_numberOfTRTHits',
 'p_numberOfTRTXenonHits',
 'p_phi',
 'p_ptPU30',
 'p_pt_track',
 'p_ptcone20',
 'p_ptcone30',
 'p_ptcone40',
 'p_sigmad0',
 'p_vertex',
 'p_weta2',
 'p_z0']

In [3]:


# 1. Load data
df = pd.read_parquet("AppML_InitialProject_train.parquet.gz")

# 2. Split into X, y_class, y_reg
X = df.drop(columns=["p_Truth_isElectron", "p_truth_Energy"])
y_class = df["p_Truth_isElectron"]
y_reg   = df.loc[y_class==1, "p_truth_Energy"]
X_reg   = X.loc[y_class==1]

# 3. Train/test split for internal evaluation
Xc_train, Xc_val, yc_train, yc_val = train_test_split(X, y_class, test_size=0.2, stratify=y_class, random_state=42)
Xr_train, Xr_val, yr_train, yr_val = train_test_split(X_reg, y_reg, test_size=0.2, random_state=42)

# 4. Define preprocessing for numeric (and categorical if any)
numeric_feats = [  # up to 25 for classification
    'p_eta', 'p_pt_track', # … add your chosen features
]
numeric_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
])

# 5. Classification pipeline
clf_pipeline = Pipeline([
    ('preproc', ColumnTransformer([
        ('num', numeric_transformer, numeric_feats),
        # ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_feats),
    ])),
    ('selector', SelectKBest(mutual_info_classif, k=25)),
    ('clf', GridSearchCV(
        estimator=RandomForestClassifier(random_state=42),
        param_grid={
            'clf__n_estimators': [100, 300],
            'clf__max_depth': [None, 10, 20],
        },
        scoring='neg_log_loss',
        cv=5,
        n_jobs=-1,
    )),
])

# 6. Regression pipeline (only true electrons)
reg_pipeline = Pipeline([
    ('preproc', ColumnTransformer([
        ('num', numeric_transformer, numeric_feats[:12]),
    ])),
    ('selector', SelectKBest(f_regression, k=12)),
    ('reg', GridSearchCV(
        estimator=MLPRegressor(random_state=42, max_iter=500),
        param_grid={
            'reg__hidden_layer_sizes': [(50,), (100,)],
            'reg__alpha': [0.0001, 0.001],
        },
        scoring='neg_mean_absolute_error',
        cv=5,
        n_jobs=-1,
    )),
])

# 7. Fit & evaluate
clf_pipeline.fit(Xc_train, yc_train)
y_proba = clf_pipeline.predict_proba(Xc_val)[:,1]
print("LogLoss (val):", log_loss(yc_val, y_proba))

reg_pipeline.fit(Xr_train, yr_train)
y_pred = reg_pipeline.predict(Xr_val)
print("Rel. MAE (val):", mean_absolute_error(yr_val, y_pred) / yr_val.mean())

# 8. Apply to test sets
X_test_clf = pd.read_parquet("AppML_InitialProject_test_classification.parquet.gz")
probs = clf_pipeline.predict_proba(X_test_clf)[:,1]

X_test_reg = pd.read_parquet("AppML_InitialProject_test_regression.parquet.gz")
# filter test electrons if known subset isn’t provided, else run on all
energy_pred = reg_pipeline.predict(X_test_reg)

# 9. Write out submission files
pd.DataFrame({'index': range(len(probs)), 'p_electron_prob': probs}) \
    .to_csv("Classification_YourName_RF1.csv", index=False)
pd.DataFrame({'feature': numeric_feats[:25]}) \
    .to_csv("Classification_YourName_RF1_VariableList.csv", index=False)

pd.DataFrame({'index': range(len(energy_pred)), 'energy_pred_GeV': energy_pred}) \
    .to_csv("Regression_YourName_MLP1.csv", index=False)
pd.DataFrame({'feature': numeric_feats[:12]}) \
    .to_csv("Regression_YourName_MLP1_VariableList.csv", index=False)


ImportError: Unable to find a usable engine; tried using: 'pyarrow', 'fastparquet'.
A suitable version of pyarrow or fastparquet is required for parquet support.
Trying to import the above resulted in these errors:
 - Missing optional dependency 'pyarrow'. pyarrow is required for parquet support. Use pip or conda to install pyarrow.
 - Missing optional dependency 'fastparquet'. fastparquet is required for parquet support. Use pip or conda to install fastparquet.